# Simulating parameterised ZX-diagrams: demonstrations

Each section below runs one of the changes described in `param-simulation-changes.md` and checks it against the **exact tensor** of the corresponding concrete diagram. Run it top to bottom; every check ends in an `assert`.

In [1]:
import sys, itertools, random
sys.path.insert(0, '..')  # use this checkout of pyzx
import numpy as np
import pyzx as zx
from fractions import Fraction
from pyzx.graph.scalar import Scalar
from pyzx.symbolic import Poly, new_var

def amplitude(g):
    """Exact value of a closed diagram, by tensor contraction."""
    return complex(np.asarray(g.to_tensor()).flatten()[0])

def circuit(q, d, seed, input_phases):
    """A random closed Clifford+T circuit, with input i given phase input_phases[i]."""
    g = zx.generate.cliffordT(q, d, seed=seed)
    g.apply_state('0' * q)
    g.apply_effect('0' * q)
    for v, p in enumerate(input_phases):
        g.set_phase(v, p)
    return g

def evaluate(scalars, vals):
    """Sum of the term scalars for an assignment {name: 0/1}."""
    return sum(complex(s.substitute_variables({v: vals[v.name] for v in s.free_vars()}).to_number()) for s in scalars)

## 1. Simulating a parameterised diagram

A 5-qubit, depth-200 Clifford+T circuit whose inputs are Boolean parameters `a`–`e`. `simulate` returns the term scalars, whose sum is the amplitude as a function of the parameters. For every one of the 32 assignments this matches simulating the concrete circuit, for both strategies, and every term is a pure scalar.

In [2]:
q, d, seed = 5, 200, 42
names = 'abcde'
params = [new_var(n, True) for n in names]

for strat in (zx.simulation.Strategy.BSS, zx.simulation.Strategy.MAGIC_CAT):
    random.seed(0)
    terms = zx.simulation.full_decompose(strat, circuit(q, d, seed, params))
    random.seed(0)
    res = zx.simulation.simulate(strat, circuit(q, d, seed, params))
    worst = 0
    for bits in itertools.product([0, 1], repeat=q):
        exact = amplitude(circuit(q, d, seed, [Fraction(b) for b in bits]))
        worst = max(worst, abs(evaluate(res, dict(zip(names, bits))) - exact))
    print(f"{strat.name}: {len(res)} terms, {sum(t.num_vertices() > 0 for t in terms)} with spiders left, "
          f"worst error over 32 assignments {worst:.1e}")
    assert worst < 1e-9 and all(t.num_vertices() == 0 for t in terms)

BSS: 1029 terms, 0 with spiders left, worst error over 32 assignments 4.0e-15


MAGIC_CAT: 101 terms, 0 with spiders left, worst error over 32 assignments 7.2e-16


## 2. Why symbolic phases need care

The spider pair `(p1)—H—(p2)` with `p1` Pauli is worth `√2·e^{iπ·p1·p2}`, a formula derived for `p1` in `[0, 2)`. Take `p1 = c + 1`, `p2 = 3/2`. At `c = 1`, `p1 = 2 ≡ 0`, so the pair is worth `+√2`. Multiplying out `(c+1)·3/2` gives `e^{3iπ} = −√2` instead. Rewriting `c + 1` as `1 − c` (the same phase, but in range) gives the right answer.

In [3]:
c = new_var('c', True)
for cv in (0, 1):
    p1 = (Fraction(cv) + 1) % 2
    g = zx.Graph()
    u = g.add_vertex(zx.VertexType.Z, 0, 0, p1); w = g.add_vertex(zx.VertexType.Z, 0, 1, Fraction(3, 2))
    g.add_edge((u, w), zx.EdgeType.HADAMARD)
    expanded = np.sqrt(2) * np.exp(1j * np.pi * float((cv + 1) * Fraction(3, 2)))
    in_range = np.sqrt(2) * np.exp(1j * np.pi * float((1 - cv) * Fraction(3, 2)))
    print(f"c={cv}: true {amplitude(g):.3f} | (c+1)*3/2: {expanded:.3f} | (1-c)*3/2: {in_range:.3f}")

c=0: true -0.000-1.414j | (c+1)*3/2: -0.000-1.414j | (1-c)*3/2: -0.000-1.414j
c=1: true 1.414-0.000j | (c+1)*3/2: -1.414+0.000j | (1-c)*3/2: 1.414+0.000j


## 3. Phase pairs: products of symbolic phases stay factored

A random spider pair with 4 Boolean parameters in each phase. Before this change the product was multiplied out into the global phase polynomial (reproduced here with `p1 * p2`). Now it is a single phase pair, whose factors are reduced into `[0, 2)` only after substitution. The cell checks both against the exact value for all 256 assignments.

In [4]:
v = {n: new_var(n, True) for n in 'abcdefgh'}
examples = {
    'Pauli x Pauli':  (v['a'] + 1 + v['b'] + v['c'] + v['d'], v['e'] + v['f'] + v['g'] + v['h']),
    'Pauli x T-like': (v['a'] + 1 + v['b'] + v['c'] + v['d'], v['e'] + Fraction(3, 4) + v['f'] + v['g'] + v['h']),
}
for label, (p1, p2) in examples.items():
    new = Scalar(); new.add_spider_pair(p1, p2)
    old = Scalar(); old.add_power(1); old.add_phase(p1 * p2)   # the previous behaviour
    wrong_old = wrong_new = 0
    for bits in itertools.product([0, 1], repeat=8):
        vals = dict(zip('abcdefgh', bits))
        g = zx.Graph()
        a = g.add_vertex(zx.VertexType.Z, 0, 0, p1); b = g.add_vertex(zx.VertexType.Z, 0, 1, p2)
        g.add_edge((a, b), zx.EdgeType.HADAMARD)
        exact = amplitude(g.substitute_variables(vals))
        wrong_old += abs(evaluate([old], vals) - exact) > 1e-9
        wrong_new += abs(evaluate([new], vals) - exact) > 1e-9
    print(f"{label}:  ({p1}) -H- ({p2})")
    print(f"   before: {len(old.phase.terms)} terms, wrong for {wrong_old}/256 | {old}")
    print(f"   now:    {len(new.phasepairs)} phase pair, wrong for {wrong_new}/256 | {new}")
    assert wrong_new == 0

Pauli x Pauli:  (a + 1 + b + c + d) -H- (e + f + g + h)
   before: 20 terms, wrong for 0/256 | sqrt(2)^1exp((a⋅e + a⋅f + a⋅g + a⋅h + e + f + g + h + b⋅e + b⋅f + b⋅g + b⋅h + c⋅e + c⋅f + c⋅g + c⋅h + d⋅e + d⋅f + d⋅g + d⋅h)ipi)
   now:    1 phase pair, wrong for 0/256 | sqrt(2)^1exp([e + f + g + h]⋅[a + 1 + b + c + d]ipi)
Pauli x T-like:  (a + 1 + b + c + d) -H- (e + 3/4 + f + g + h)
   before: 25 terms, wrong for 240/256 | sqrt(2)^1exp((a⋅e + 3/4⋅a + a⋅f + a⋅g + a⋅h + e + 3/4 + f + g + h + b⋅e + 3/4⋅b + b⋅f + b⋅g + b⋅h + c⋅e + 3/4⋅c + c⋅f + c⋅g + c⋅h + d⋅e + 3/4⋅d + d⋅f + d⋅g + d⋅h)ipi)
   now:    1 phase pair, wrong for 0/256 | sqrt(2)^1exp([a + 1 + b + c + d]⋅[e + 3/4 + f + g + h]ipi)


Products that can be folded exactly without growing the phase polynomial still are: concrete factors, integer multipliers, and factors with at most one variable.

In [5]:
a, b = new_var('a', True), new_var('b', True)
for p1, p2 in [(Fraction(1), Fraction(3, 4)), (a, b), (a + 1, Fraction(3, 2))]:
    s = Scalar(); s.add_spider_pair(p1, p2)
    print(f"({p1}) -H- ({p2}):  {s}   [phase pairs: {len(s.phasepairs)}]")

(1) -H- (3/4):  -1.00+1.00i = sqrt(2)^1exp(3/4ipi)   [phase pairs: 0]
(a) -H- (b):  sqrt(2)^1exp(a⋅bipi)   [phase pairs: 0]
(a + 1) -H- (3/2):  sqrt(2)^1exp((1/2⋅a + 3/2)ipi)   [phase pairs: 0]


## 4. Parameterised scalar diagrams reduce fully

lcomp now applies to a phase that is always ±π/2 (e.g. `a + 1/2`), and pivot to symbolic Pauli phases (e.g. `c + 1`), but only in diagrams without inputs or outputs. Open diagrams keep their symbolic spiders, e.g. measurement outcomes.

In [6]:
from pyzx.rewrite_rules.lcomp_rule import check_lcomp
from pyzx.rewrite_rules.pivot_rule import check_pivot

def triangle(centre_phase, leaf_phases, open_diagram=False):
    g = zx.Graph()
    centre = g.add_vertex(zx.VertexType.Z, 0, 0, centre_phase)
    leaves = [g.add_vertex(zx.VertexType.Z, 1, i, p) for i, p in enumerate(leaf_phases)]
    for i, u in enumerate(leaves):
        g.add_edge((centre, u), zx.EdgeType.HADAMARD)
        g.add_edge((u, leaves[(i + 1) % 3]), zx.EdgeType.HADAMARD)
    if open_diagram:
        out = g.add_vertex(zx.VertexType.BOUNDARY, 2, 0); g.add_edge((leaves[0], out)); g.set_outputs((out,))
    return g, centre, leaves

a, c, d = new_var('a', True), new_var('c', True), new_var('d', True)
for open_diagram in (False, True):
    g, centre, leaves = triangle(a + Fraction(1, 2), [Fraction(1, 4), c, Fraction(3, 4)], open_diagram)
    h, centre2, leaves2 = triangle(c + 1, [d, Fraction(1, 4), Fraction(3, 4)], open_diagram)
    print(f"{'open' if open_diagram else 'closed'} diagram: lcomp on a+1/2 applies: {check_lcomp(g, centre)} | "
          f"pivot on (c+1, d) applies: {check_pivot(h, centre2, leaves2[0])}")

closed diagram: lcomp on a+1/2 applies: True | pivot on (c+1, d) applies: True
open diagram: lcomp on a+1/2 applies: False | pivot on (c+1, d) applies: False


## 5. MAGIC_CAT on an unreduced circuit (also affects concrete diagrams)

`check_catn` used to accept a Z spider with X neighbours on plain edges (e.g. from CNOTs), a form the cat decompositions don't implement. On this circuit, MAGIC_CAT used to give `-0.0019+0.0598j`.

In [7]:
g = circuit(5, 200, 42, [])
print("exact:    ", np.round(amplitude(g), 5))
print("MAGIC_CAT:", np.round(zx.simulation.simulate(zx.simulation.Strategy.MAGIC_CAT, g.copy()), 5))
print("BSS:      ", np.round(zx.simulation.simulate(zx.simulation.Strategy.BSS, g.copy()), 5))
assert abs(zx.simulation.simulate(zx.simulation.Strategy.MAGIC_CAT, g.copy()) - amplitude(g)) < 1e-9

centre_graph = zx.Graph()
centre = centre_graph.add_vertex(zx.VertexType.Z, 0, 0, 0)
for i in range(3):
    w = centre_graph.add_vertex(zx.VertexType.X, 1, i, Fraction(1, 4)); centre_graph.add_edge((centre, w))
try:
    zx.simulation.check_valid(zx.simulation.Decomp.CAT_3, centre_graph, centre)
except ValueError as e:
    print("opposite-colour form rejected:", e)

exact:     (-0.16115-0.07276j)
MAGIC_CAT: (-0.16115-0.07276j)


BSS:       (-0.16115-0.07276j)
opposite-colour form rejected: The cat 3 decomposition must act on a Z spider with 3 Z-spider neighbours connected by Hadamard edges.


## 6. Cat decompositions with a π centre

All four cat decompositions were wrong for a π-phase centre spider. They are now built from `cat_π = e^{iπ/4}·X·S†·cat_0` on one leg.

In [8]:
from pyzx.simulation import Decomp, apply_decomp
for n, decomp in [(3, Decomp.CAT_3), (4, Decomp.CAT_4), (5, Decomp.CAT_5), (6, Decomp.CAT_6)]:
    row = []
    for centre_phase in (0, 1):
        g = zx.Graph(); centre = g.add_vertex(zx.VertexType.Z, 1, 0, centre_phase); outs = []
        for i in range(n):
            w = g.add_vertex(zx.VertexType.Z, 2, i, Fraction(1, 4)); g.add_edge((centre, w), zx.EdgeType.HADAMARD)
            b = g.add_vertex(zx.VertexType.BOUNDARY, 3, i); g.add_edge((w, b)); outs.append(b)
        g.set_outputs(tuple(outs))
        ok = np.allclose(g.to_tensor(), apply_decomp(decomp, g, centre).to_tensor())
        row.append(f"centre {centre_phase}: {'ok' if ok else 'WRONG'}")
        assert ok
    print(f"cat{n}:", " | ".join(row))

cat3: centre 0: ok | centre 1: ok
cat4: centre 0: ok | centre 1: ok
cat5: centre 0: ok | centre 1: ok


cat6: centre 0: ok | centre 1: ok


## 7. Empty polynomials and `sum_of_phases` keys

A `Poly` whose terms cancel is now falsy, like 0. A `sum_of_phases` term whose phase cancels is merged with the phase-0 term.

In [9]:
a = new_var('a', True)
print("bool((a+1) - (a+1)) =", bool((a + 1) - (a + 1)))
s = Scalar()
s.multiply_sum_of_phases({0: 1, a: 1})
s.multiply_sum_of_phases({0: 1, a: -1})   # (1 + e^{i pi a})(1 - e^{i pi a}) = 1 - e^{2 i pi a} = 0
print("sum_of_phases:", s.sum_of_phases)

bool((a+1) - (a+1)) = False
sum_of_phases: {}


## 8. Non-Boolean parameters

Continuous parameters take the same rewrite paths as before. The widened lcomp/pivot matching and the phase-pair folding only treat Boolean phases specially. Results are correct, and where a continuous-phase spider can't be removed by the T-decompositions, simulation stops with an explicit error rather than returning something wrong.

In [10]:
rng = random.Random(1)
wrong = 0
for seed in range(10):
    x = new_var('x', False); y = new_var('y', False); b = new_var('b', True)
    g = circuit(3, 60, seed, [x, y, b])
    h = g.copy(); zx.simplify.full_reduce(h)
    for _ in range(3):
        vals = {'x': Fraction(rng.randint(0, 2000), 1000), 'y': Fraction(rng.randint(0, 2000), 1000), 'b': rng.randint(0, 1)}
        wrong += not np.allclose(g.substitute_variables(vals).to_tensor(), h.substitute_variables(vals).to_tensor())
print("full_reduce with continuous + Boolean parameters, wrong evaluations:", wrong, "of 30")
assert wrong == 0

for seed in range(6):
    g = circuit(3, 40, seed, [new_var('x', False)])
    try:
        res = zx.simulation.simulate(zx.simulation.Strategy.MAGIC_CAT, g.copy())
    except RuntimeError as e:
        print(f"seed {seed}: explicit error: {e}")
        continue
    # A plain number means x dropped out of every term, i.e. the amplitude doesn't depend on x
    value = lambda xv: res if not isinstance(res, list) else \
        sum(complex(s.substitute_variables({u: xv for u in s.free_vars()}).to_number()) for s in res)
    ok = all(abs(value(xv) - amplitude(g.substitute_variables({'x': xv}))) < 1e-9
             for xv in (Fraction(0), Fraction(7, 10), Fraction(13, 10)))
    kind = f"{len(res)} terms" if isinstance(res, list) else "a number (x drops out)"
    print(f"seed {seed}: returned {kind}, correct at x = 0, 0.7, 1.3: {ok}")
    assert ok

full_reduce with continuous + Boolean parameters, wrong evaluations: 0 of 30
seed 0: returned a number (x drops out), correct at x = 0, 0.7, 1.3: True
seed 1: returned 3 terms, correct at x = 0, 0.7, 1.3: True
seed 2: returned 2 terms, correct at x = 0, 0.7, 1.3: True
seed 3: returned 2 terms, correct at x = 0, 0.7, 1.3: True
seed 4: returned 2 terms, correct at x = 0, 0.7, 1.3: True
seed 5: returned 3 terms, correct at x = 0, 0.7, 1.3: True
